Cell 1 - Imports

In [1]:
from pathlib import Path
import re
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

Cell 2 — User settings

In [ ]:
# Folder containing predicted tile masks
PRED_DIR = Path("./output/test_predictions_a10_segment1")   # change if needed

# Output files
OUT_MASK_PATH = PRED_DIR / "stitched_prediction_mask.png"
OUT_COLOR_PATH = PRED_DIR / "stitched_prediction_color.png"

# Class colors for visualization
CLASS_COLORS = np.array([
    [0, 0, 0],        # 0 = background
    [0, 255, 0],      # 1 = good
    [255, 255, 0],    # 2 = fair
    [255, 0, 0]       # 3 = poor
], dtype=np.uint8)

Cell 3 — Helper: extract x, y, tile size from filename

In [ ]:
def parse_tile_filename(filename):
    """
    Extract x, y, s from filename.
    Example:
        x1024_y1536_s512.png
        pred_mask_x1024_y1536_s512.png
    Returns:
        (x, y, s)
    """
    m = re.search(r"x(\d+)_y(\d+)_s(\d+)", filename)
    if m is None:
        raise ValueError(f"Could not parse tile coordinates from filename: {filename}")
    
    x = int(m.group(1))
    y = int(m.group(2))
    s = int(m.group(3))
    return x, y, s

Cell 4 — List prediction files

In [ ]:
pred_files = sorted([
    p for p in PRED_DIR.glob("*.png")
    if p.name != OUT_MASK_PATH.name and p.name != OUT_COLOR_PATH.name
])

print("Number of predicted tiles found:", len(pred_files))
print("Example file:", pred_files[0].name if pred_files else "None")

Cell 5 — Determine output canvas size

In [ ]:
tile_info = []

for p in pred_files:
    x, y, s = parse_tile_filename(p.name)
    tile_info.append((p, x, y, s))

# Compute stitched image size
max_x = max(x + s for _, x, y, s in tile_info)
max_y = max(y + s for _, x, y, s in tile_info)

print("Stitched mask size will be:")
print("Width :", max_x)
print("Height:", max_y)

Cell 6 — Stitch tiles into one full mask

If two overlapping tiles predict the same class, averaging keeps that class.

In [ ]:
OVERLAP = 128   # change to your overlap
TILE_SIZE = 512

stitched_sum = np.zeros((max_y, max_x), dtype=np.float32)
stitched_count = np.zeros((max_y, max_x), dtype=np.float32)

for p, x, y, s in tile_info:
    tile = np.array(Image.open(p))

    # If tile is RGB by mistake, take first channel
    if tile.ndim == 3:
        tile = tile[..., 0]

    h, w = tile.shape[:2]

    # Add tile values
    stitched_sum[y:y+h, x:x+w] += tile.astype(np.float32)
    stitched_count[y:y+h, x:x+w] += 1.0

# Avoid division by zero
stitched_count[stitched_count == 0] = 1.0

# Average overlapping predictions
stitched_mask = np.rint(stitched_sum / stitched_count).astype(np.uint8)

print("Stitching complete.")
print("Stitched mask shape:", stitched_mask.shape)
print("Unique classes in stitched mask:", np.unique(stitched_mask))

Or the more correct overlap-aware version, 

For overlap = 128:

- crop 64 px from each internal tile edge

- keep full border only on outermost edges

In [ ]:
OVERLAP = 128
HALF_OVERLAP = OVERLAP // 2

stitched_mask = np.zeros((max_y, max_x), dtype=np.uint8)

# Determine outer bounds
min_x = min(x for _, x, y, s in tile_info)
min_y = min(y for _, x, y, s in tile_info)
max_tile_x = max(x for _, x, y, s in tile_info)
max_tile_y = max(y for _, x, y, s in tile_info)

for p, x, y, s in tile_info:
    tile = np.array(Image.open(p))

    if tile.ndim == 3:
        tile = tile[..., 0]

    h, w = tile.shape[:2]

    # crop margins
    crop_left = 0 if x == min_x else HALF_OVERLAP
    crop_top = 0 if y == min_y else HALF_OVERLAP
    crop_right = 0 if x == max_tile_x else HALF_OVERLAP
    crop_bottom = 0 if y == max_tile_y else HALF_OVERLAP

    tile_cropped = tile[
        crop_top:h-crop_bottom if crop_bottom > 0 else h,
        crop_left:w-crop_right if crop_right > 0 else w
    ]

    y_start = y + crop_top
    x_start = x + crop_left
    y_end = y_start + tile_cropped.shape[0]
    x_end = x_start + tile_cropped.shape[1]

    stitched_mask[y_start:y_end, x_start:x_end] = tile_cropped

print("Stitching complete.")
print("Stitched mask shape:", stitched_mask.shape)
print("Unique classes in stitched mask:", np.unique(stitched_mask))

Cell 7 — Save stitched integer mask

In [ ]:
Image.fromarray(stitched_mask).save(OUT_MASK_PATH)
print("Saved stitched integer mask to:", OUT_MASK_PATH)

Cell 8 — Convert stitched mask to color and save

In [ ]:
stitched_color = CLASS_COLORS[stitched_mask]

Image.fromarray(stitched_color).save(OUT_COLOR_PATH)
print("Saved stitched color mask to:", OUT_COLOR_PATH)

Cell 9 — Quick visualization

In [ ]:
plt.figure(figsize=(10, 10))
plt.imshow(stitched_color)
plt.title("Stitched Predicted Mask")
plt.axis("off")
plt.show()

Cell 10 — Overlay stitched prediction on original pseudo-ortho

In [ ]:
# Change this path to your original full-site image if you want overlay
ORIGINAL_IMAGE_PATH = Path("./output/pseudo_ortho/a58_segment2/pseudo_ortho_rotated.preview.png")

In [ ]:
if ORIGINAL_IMAGE_PATH.exists():
    original = np.array(Image.open(ORIGINAL_IMAGE_PATH).convert("RGB"))

    # Make sure dimensions match
    H = min(original.shape[0], stitched_color.shape[0])
    W = min(original.shape[1], stitched_color.shape[1])

    alpha = 0.4
    overlay = (
        (1 - alpha) * original[:H, :W].astype(np.float32) +
        alpha * stitched_color[:H, :W].astype(np.float32)
    ).astype(np.uint8)

    plt.figure(figsize=(10, 10))
    plt.imshow(overlay)
    plt.title("Overlay: Original Image + Stitched Prediction")
    plt.axis("off")
    plt.show()
else:
    print("Original image not found, skipping overlay.")